# Laboratory 2 - Report
Remigiusz Babiarz
# Calculating the current flowing through a circuit represented as a graph 

## Convention and general assumptions
As the input is a graph that is meant to be representing a circuit it is important to make some assumptions and prepare a convention as to how to represent its parameters. 
I assume every graph to be undirected, and that at no point will I create any structure representing a directed graph as I believe that to be unnecessary. 
Nodes are numbers ranging from $0$ to $N-1$ where $N$ is the number of nodes and possess no attributes.
Edge attributes used to represent various circuit parameters:
- $R$ - resistance of an edge,
- $G$ - conductance of an edge,
- $I$ - current flowing through an edge.

$R$ and $G$ are to be set once during edge creation. $I$ is to be equal to $0$ upon creation and will be assigned a value by a current calculating algorithm. \
As the graph is undirected the following convention for current assignment is assumed:
- current with a positive value $I>0$ flows from the node with higher index into node with lower index,
- current with a negative value $I<0$ flows from the node with lower index into node with higher index.

Throught the report there is also the attribute $point$ being assigned to nodes - its purpose is solely for custom indexing of nodes in visualization. 

## Useful imports

In [275]:
import numpy as np
import random
import matplotlib.pyplot as plt
import time
import copy
import math
import networkx as nx
import ipycytoscape

## Visualization utils
Decidied to use ```ipycytoscape``` for generating interactive widgets. Various visualization functions described below.

In [276]:
# produces an interactive widget representing a circuit with numbers on edges representing resistance
def show_circuit(CG):
    cytoscapeobj = ipycytoscape.CytoscapeWidget()
    cytoscapeobj.graph.add_graph_from_networkx(CG)
    cytoscapeobj.set_style([
    {
        'selector': 'node',
        'css': {
            'content': 'data(point)', 
            'text-valign': 'center',
            'color': 'black',
            'background-color': 'lightgray'
        }
    },
    {
        'selector': 'edge',
        'css': {
            'content': 'data(R)', 
            'font-size': '12px',
            'text-background-color': 'white',
            'text-background-opacity': 0.8
        }
    }
    ])
    display(cytoscapeobj)

# produces an interactive widget representing a circuit with arrows labeled with the current flowing through them 
# arrow opacity translates to current relative to maximum
def show_solved(CG):
    vis_graph = nx.DiGraph()
    
    for n, d in CG.nodes(data=True):
        node_data = d.copy()
        vis_graph.add_node(n, **node_data)
        
    max_I = max([abs(round(d.get('I', 0), 2)) for _, _, d in CG.edges(data=True)], default=1)
    if max_I == 0: max_I = 1 
        
    for u, v, d in CG.edges(data=True):
        raw_I = d.get('I', 0)
        I = round(raw_I, 2)
        
        node_u = max(u, v) 
        node_v = min(u, v)
        
        if I < 0:
            src, tgt = node_v, node_u
        else:
            src, tgt = node_u, node_v
            
        edge_data = d.copy()
        edge_data['I'] = I 
        edge_data['abs_I'] = abs(I)
        vis_graph.add_edge(src, tgt, **edge_data)

    cytoscapeobj = ipycytoscape.CytoscapeWidget()
    cytoscapeobj.graph.add_graph_from_networkx(vis_graph)
    
    cytoscapeobj.set_style([
        {
            'selector': 'node',
            'css': {
                'content': 'data(point)', 
                'text-valign': 'center',
                'text-wrap': 'wrap', 
                'font-size': '10px',
                'color': 'black',
                'background-color': 'lightgray',
                'width': '40px',  
                'height': '40px'
            }
        },
        {
            'selector': 'edge',
            'css': {
                'content': 'data(abs_I)', 
                'font-size': '10px',
                'text-background-color': 'white',
                'text-background-opacity': 0.8,
                'curve-style': 'bezier',
                'target-arrow-shape': 'triangle',
                'line-color': f'mapData(abs_I, 0, {max_I}, #a1d99b, #005a32)', 
                'target-arrow-color': f'mapData(abs_I, 0, {max_I}, #a1d99b, #005a32)',
                'width': f'mapData(abs_I, 0, {max_I}, 1.5, 6)',
                'opacity': f'mapData(abs_I, 0, {max_I}, 0.3, 1.0)'
            }
        }
    ])
    
    display(cytoscapeobj)

# displays a matrix in console - used only for debug purposes 
def print_matrix(X):
    for row in X:
        for x in row:
            print(round(x,2), end='\t')
        print()

## Loading circuits, generating graphs

In [277]:
# loads a circuit from file
# expected file structure:
# {number of nodes}
# {s} {t} {E}
# {edge list}
# example - cycle with length 3:
# 3
# 0 2 10
# 0 1 4
# 1 2 3
# 2 0 1
def load_circuit_from_file(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    N = int(lines[0].strip())

    s, t, E = lines[1].split()
    s = int(s)
    t = int(t)
    E = float(E)
    CG = nx.Graph()
    for i in range(N):
        CG.add_node(i,point=i, V = 0)
    for v, u, r in [tuple(l.split()) for l in lines[2:]]:
        CG.add_edge(int(v), int(u), R=float(r), G=1/float(r), I = 0.0)

    return CG, s, t, E

# adds circuit parameters to a graph - such as resistance and current 
def setup_circuit(graph, Rfun = lambda: 0):
    nx.set_edge_attributes(graph, 0, 'I')
    for edge in nx.edges(graph):
        R =  Rfun()
        graph[edge[0]][edge[1]]['R'] = R
        graph[edge[0]][edge[1]]['G'] = 1/R
    nx.set_node_attributes(graph, dict([i, i] for i in range(len(graph.nodes))), 'point' )
    return graph

# creates a random graph using the erdos_renyi probabilistic model
def random_erdos(N, max_tries = 50):
    # it can be proven that if edge creation probability beyond the threshhold: 
    # (math.log(N)+p(N))/N, where p(N) is any function that approaches infinity as N approaches infinity
    # then generated graphs are almost certainly fully connected
    p = (math.log(N) + 0.01*N)/N
    tries = 0
    
    # still, retry until created graph is connected
    while not nx.is_connected(G := nx.erdos_renyi_graph(N,p)) and tries < max_tries: tries += 1
    
    if not nx.is_connected(G): return None
    return G

# creates a graph being two random graphs connected with a bridge
def random_double_erdos(N, max_tries = 50):
    if N % 2 == 1: N+=1
    G1 = random_erdos(N//2)
    G2 = random_erdos(N//2)
    G = nx.disjoint_union(G1, G2)
    G.add_edge(N//2-1, N//2)
    return G

# 2d grid graph with the dimentions a and b
def grid_2d(a, b):
    G = nx.grid_2d_graph(a, b)
    G = nx.convert_node_labels_to_integers(G)
    return G

# cubical graph with 8 nodes and 12 edges 
def cubical():
    G = nx.cubical_graph()
    return G

# a small world graph identified by the following property:
# the shortest path between any two nodes is short (approx < 6 nodes)
# the parameters have been chosen experimentally
def small_world(N):
    k = N//5
    p = 0.05
    G = nx.watts_strogatz_graph(N,k,p)
    return G



## Nodal Analysis (analiza potencjałów węzłowych)

The method relies on choosing a ground node, assigning variables representing the potential of each node and then constructing a set of equations, each representing Kirhoff's first law for a given node. This equation system may be rewritten in matrix form:
$$ G \cdot V = Y $$
where $G$ is the conductance matrix constructed in the following way:
- $G_{i,i} = \Sigma_e \text{conductance of e}$, where e is the edge (resistor) adjecent to the node indexed $i$,
- $G_{i,j} = -\text{conductance of edge } (i,j)$ (or 0, if the edge does not exist), \

$V$ is the vector of potentials, \
$Y$ is the vector of independent sources of potential, in this case: $\forall _i : Y_i = 0$. 

Since we already know the voltage on the added edge connecting the nodes $s$ and $t$ (the source), and we can choose our reference (ground) node as we please we can make some significant simplifications:
1. Let $V_s = E$, $V_t = 0$,
2. Remove all terms containing $V_t$ (as it's equal to 0),
3. Move all terms containing only $V_s$ as the potential to the right hand side of the equation (as it's constant and equal to E),
4. Construct the matrix again.

After performing those steps we are left with the same matrix $X$ and vector $V$, but with dimensions $N-2$ instead of $N$, as we have eliminated the constants $V_s$ and $V_t$. The lab instruction mentions describing the solution to the overdetermined system of equations - my solution is not constructing one such as that in the first place. \
Vector Y becomes:
$$ Y_{i} = \begin{cases}
   E \cdot G_{i,s} &\text{if  there exists edge} (i,s) \\
   0 &\text{otherwise} 
\end{cases} $$ 

After solving this system of equations using `numpy`, currents can be calculated using Ohm's law.


In [278]:
'''
params:
G -> networkx graph representing a circuit
s, t -> two nodes with a voltage of E between them
E -> voltage
'''
def nodal_analysis(CGin : nx.Graph, s : int, t : int, E : float):
    CG = CGin.copy()
    N = len(CG.nodes)
    # setting up matrices
    X = np.zeros((N-2,N-2))
    Y = np.zeros(N-2)

    # we already know 2 of N voltages - create list of unknowns
    unk = list(filter(lambda i : i != s and i != t, range(N)))

    # construct matrix made out of equations - one for each unknown potential
    for i in range(N-2):
        for j in range(N-2):
            if i == j:
                X[i][i] = sum(list(map(lambda edge : CG[edge[0]][edge[1]]["G"], CG.edges(unk[i]))))
            else:
                X[i][j] = -CG[unk[i]][unk[j]]["G"] if CG.has_edge(unk[i], unk[j]) else 0 

    for i in range(N-2):
        Y[i] = CG[unk[i]][s]["G"]*E if CG.has_edge(unk[i], s) else 0 

    # calculate unknown potentials    
    Vunk = list(np.linalg.solve(X, Y))
    
    # construct the final potential list
    V = [0.0] * N 
    V[s] = E 
    V[t] = 0.0
    for idx, unk_node in enumerate(unk):
        V[unk_node] = Vunk[idx]
    
    # map currents onto edges using Ohm's law
    # current direction is determined by previously established convention 
    for i in range(N):
        for j in range(i+1, N):
            if (i,j) not in CG.edges: continue
            U = V[j]-V[i]
            I = float(CG[i][j]["G"] * abs(U))
            CG[i][j]["I"] = I*np.sign(U)
            
    return CG


## Loop Analysis (z użyciem praw Kirhoff'a)

While looking for sources on how to perform this method on a circuit I mostly found descriptions of the mesh circuit analysis method, which relies on the circuit in question being planar. When considering random graphs, we of course cannot guarantee planarity. 

In that case, let's construct the equations step by step. 

Equations are derived form Kirhoff's second law. To construct those we of course need to find all loops within the circuit. This itself could be a problem, as it's not an easy task to find all needed cycles on an arbitrary graph. Even if an appropriate method method is found and the cycles it produces cover all edges through which the current flows, we may find ourselves with unnecessary cycles - and what follows - unneccesary equations. 

One solution I found while researching this problem was using the cycle basis of a graph. A cycle basis is a set of cycles that has the following property: any cycle within the graph can be constructed using cycles from the basis. `networkx` provides the function `cycle_basis(G)` which returns the *minimal* cycle basis of a graph - that is the smallest basis in the sense of amount of cycles within the set. This also implies that no cycle within the basis can be constructed using any number of other cycles from the basis - this property guarantees that the number of equations derived from this method is minimal.

While in the previous method including the voltage $E$ between nodes $s$ and $t$ as trivial, in this case it requires a bit more work. A new edge is created between $s$ and $t$ with resistance $0$ and a voltage source with $V=E$.

We not can begin constructing the system of equations derived from Kirhoff's second law - one equation per loop (cycle). That system represents how the currents within the loops interact with each other while abiding current laws.

A singular equation within the system of $N$ equations, where N is the number of loops, looks as such:

$$ I_j \cdot R_{j,j} + \Sigma_k (I_k \cdot R_{j,k}) = U_j $$

where:
- $I_i$ - current flowing within loop $i$,
- $R_{j,j}$ - sum of resistances on the loop $i$,
- $U_j$ - sum of voltage sources on the loop $i$,
- $R_{j,k}$ - sum of resistances shared between loops $j$ and $k$. 

One problem that arises that isn't immediately apparent is determining the factor $R_{j,k}$. When the current $I_k$ flows in the same direction through the resistor as the current $I_j$, it increases the current, while in the opposite case - counteracts it. This means that when calculating the shared resistance we need to consider the relative direction of current within the loops and depending on it either substact or add the term $I_k \cdot R_{j,k}$.

This system of equations can be rewritten in matrix form:
$$ R \cdot I = U $$
where:
- $R$ is the resistance matrix constructed in the way mentioned above,
- $I$ is the vector of currents, \
- $E$ is the vector of independent sources of voltage.

The vector E is constructed in the following way:
$$ U_{i} = \begin{cases}
   U &\text{if within the loop i there exists edge } (s,t) \\
   0 &\text{otherwise} 
\end{cases} $$ 

The matrix equation can be solved using `numpy`. Afterwards, all currents from the loops are added to their corresponding edges.

Again, the system is not overdetermined - this property is guaranteed by the minimality of the chosen cycle basis.

In [279]:
# finds the shared resistance of two loops.
# accounts for the direction of the current
# on the edges that the currents counteract the resistance is substracted
# on the edges that the currents overlap constructively resistance is added
def shared_resistance(loop1, loop2, CG):
    R_shared = 0
    for edge in loop1:
        rev_edge = (edge[1], edge[0])
        if edge in loop2:
            R_shared += CG[edge[0]][edge[1]]["R"]
        elif rev_edge in loop2:
            R_shared -= CG[edge[0]][edge[1]]["R"]

    return R_shared 

# finds all indeces of loops that contain a given edge
def find_loops_with(v, u, loops):
    ls = []
    for i, loop in enumerate(loops):
        if (v,u) in loop or (u,v) in loop:
            ls.append(i)
    return ls

'''
params:
G -> networkx graph representing a circuit
s, t -> two nodes with a voltage of E between them
E -> voltage
'''
def loop_analysis(CGin : nx.Graph, s : int, t : int, E : float):
    CG = CGin.copy()
    # we need an additional edge to simulate E between two nodes (even if it exists)
    # networkx undirected graph doesnt allow for duplicate edges
    # therefore to add the additional edge containing the voltage E
    # we add an additional ghost node connected to the nodes s and t with resistance 0
    ghost_node = CG.number_of_nodes()
    CG.add_edge(s, ghost_node, R=0, G=float('inf'), I = 0.0)    
    CG.add_edge(ghost_node, t, R =0, G =float('inf'), I = 0.0)

    # cycle basis covers all loops
    loops = nx.cycle_basis(CG)
    N = len(loops)
    
    # convert the loops to a more desired format
    # example: 
    # [1, 2, 4] -> set((1,2), (2,4), (4,1))
    # [4, 2, 0] -> set((4,2), (2,0), (0,4))
    # set allows for O(1) lookup 
    # in above example the edge (2,4) exists in both sets. Reversal of nodes means different direction current.
    loops = [set([(loop[i-1], loop[i]) for i in range(len(loop))]) for loop in loops]
    
    # matrix setup - one equation per loop
    X = np.zeros((N, N))
    Y = np.zeros((N))
    for i in range(N):
        for j in range(N):
            if i == j:
                X[i][i] = sum(list(map(lambda edge: CG[edge[0]][edge[1]]["R"], loops[i])))
            else:
                X[i][j] = shared_resistance(loops[i], loops[j], CG)
    

    for i in find_loops_with(s, ghost_node, loops):
        Y[i] = E

    # calculating currents within loops
    I = np.linalg.solve(X, Y)

    # mapping loop currents onto the edges using previously established convention
    for i, loop in enumerate(loops):
        for edge in loop:
            CG[edge[0]][edge[1]]["I"] += I[i] * np.sign(edge[0]-edge[1])

    # removing ghost edges and node
    CG.remove_edge(s, ghost_node)
    CG.remove_edge(ghost_node, t)    
    CG.remove_node(ghost_node)
    return CG


## Testing the algorithms
Useful functions for generating test cases:

In [280]:
def test_erdos():
    results = []
G = setup_circuit(random_erdos(5), lambda: random.randrange(1, 10))
CG1 = loop_analysis(G,0,1,10)
CG2 = nodal_analysis(G,0,1,10)
show_solved(CG1)
show_solved(CG2)

CytoscapeWidget(cytoscape_layout={'name': 'cola'}, cytoscape_style=[{'selector': 'node', 'css': {'content': 'd…

CytoscapeWidget(cytoscape_layout={'name': 'cola'}, cytoscape_style=[{'selector': 'node', 'css': {'content': 'd…